In [1]:
import pandas as pd
from src.ingestion import load_retail_data

In [2]:
data = load_retail_data("data/raw")

for name, df in data.items():
    print(f"{name}: {df.shape}")

stores: (54, 5)
train: (3000888, 6)
oil: (1218, 2)
holidays_events: (350, 6)
transactions: (83488, 3)
test: (28512, 5)


In [3]:

oil = data["oil"]

print("Data types:")
print(oil.dtypes)

print("\nMissing values:")
print(oil.isna().sum())

print("\nMissing oil-price records:")
print(oil.loc[oil["dcoilwtico"].isna()].head(10))


Data types:
date              str
dcoilwtico    float64
dtype: object

Missing values:
date           0
dcoilwtico    43
dtype: int64

Missing oil-price records:
           date  dcoilwtico
0    2013-01-01         NaN
14   2013-01-21         NaN
34   2013-02-18         NaN
63   2013-03-29         NaN
104  2013-05-27         NaN
132  2013-07-04         NaN
174  2013-09-02         NaN
237  2013-11-28         NaN
256  2013-12-25         NaN
261  2014-01-01         NaN


In [4]:
from src.cleaning import convert_date_columns, filling_oil_missing_values

data = convert_date_columns(data)
oil_clean = data["oil"]

print(oil_clean.dtypes)
print("\nMissing values:")
print(oil_clean.isna().sum())

print("\nOriginal date type:", data["oil"]["date"].dtype)
print("Cleaned date type:", oil_clean["date"].dtype)

date          datetime64[us]
dcoilwtico           float64
dtype: object

Missing values:
date           0
dcoilwtico    43
dtype: int64

Original date type: datetime64[us]
Cleaned date type: datetime64[us]


In [5]:

for name, df in data.items():
    print(f"\n--- {name} ---")
    print(df.dtypes)



--- stores ---
store_nbr    int64
city           str
state          str
type           str
cluster      int64
dtype: object

--- train ---
id                      int64
date           datetime64[us]
store_nbr               int64
family                    str
sales                 float64
onpromotion             int64
dtype: object

--- oil ---
date          datetime64[us]
dcoilwtico           float64
dtype: object

--- holidays_events ---
date           datetime64[us]
type                      str
locale                    str
locale_name               str
description               str
transferred              bool
dtype: object

--- transactions ---
date            datetime64[us]
store_nbr                int64
transactions             int64
dtype: object

--- test ---
id                      int64
date           datetime64[us]
store_nbr               int64
family                    str
onpromotion             int64
dtype: object


In [6]:
data["oil"] = filling_oil_missing_values(data["oil"])

print("Missing before:", data["oil"]["dcoilwtico"].isna().sum())
print("Missing after:", data["oil"]["dcoilwtico"].isna().sum())
print(data["oil"].head())

Missing before: 0
Missing after: 0
        date  dcoilwtico
0 2013-01-01       93.14
1 2013-01-02       93.14
2 2013-01-03       92.97
3 2013-01-04       93.12
4 2013-01-07       93.20


In [7]:
from src.cleaning import cleaned_data


In [8]:
clean_data = cleaned_data(data)

print(clean_data["oil"].dtypes)
print(clean_data["oil"].isna().sum())
print(data["oil"].isna().sum())

date          datetime64[us]
dcoilwtico           float64
dtype: object
date          0
dcoilwtico    0
dtype: int64
date          0
dcoilwtico    0
dtype: int64


In [9]:

train = clean_data["train"]

print("Negative sales:", (train["sales"] < 0).sum())
print("Negative promotions:", (train["onpromotion"] < 0).sum())
print("Negative store numbers:", (train["store_nbr"] < 0).sum())

print("\nSales summary:")
print(train["sales"].describe())

print("\nPromotion summary:")
print(train["onpromotion"].describe())



Negative sales: 0
Negative promotions: 0
Negative store numbers: 0

Sales summary:
count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

Promotion summary:
count    3.000888e+06
mean     2.560006e+00
std      1.137160e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.690000e+02
Name: onpromotion, dtype: float64


In [10]:

train = clean_data["train"]

print(
    train.nlargest(10, "sales")[
        ["date", "store_nbr", "family", "sales", "onpromotion"]
    ]
)


              date  store_nbr     family       sales  onpromotion
2163723 2016-05-02          2  GROCERY I  124717.000           59
2445984 2016-10-07         39      MEATS   89576.360            0
2144154 2016-04-21         20  GROCERY I   87438.516           53
2139699 2016-04-18         45  GROCERY I   76090.000           38
2153031 2016-04-26          2  GROCERY I   63434.000           30
2145045 2016-04-21         45  GROCERY I   53874.000           44
562596  2013-11-12         44  GROCERY I   46271.000            0
1257246 2014-12-08         35  GROCERY I   45361.000            3
1916586 2015-12-14         35  GROCERY I   40351.460            7
2760297 2017-04-02          9  GROCERY I   38422.625           73


In [11]:
print(
    train.nlargest(10, "onpromotion")[
        ["date", "store_nbr", "family", "sales", "onpromotion"]
    ]
)

             date  store_nbr   family       sales  onpromotion
973002 2014-07-02          1  PRODUCE   3951.5010          169
973365 2014-07-02          2  PRODUCE   4135.4960          169
973629 2014-07-02         27  PRODUCE   3536.4300          169
974256 2014-07-02         44  PRODUCE  15184.5370          169
974289 2014-07-02         45  PRODUCE  10279.4810          169
974322 2014-07-02         46  PRODUCE   3951.1921          169
974421 2014-07-02         49  PRODUCE   8248.4590          169
974487 2014-07-02         50  PRODUCE   2931.0320          169
974685 2014-07-02          7  PRODUCE   6942.5220          169
974718 2014-07-02          8  PRODUCE   7632.3570          169


In [12]:

train = clean_data["train"]

store_53_grocery = train[
    (train["store_nbr"] == 53) &
    (train["family"] == "GROCERY I")
]

print("Number of records:", len(store_53_grocery))
print("\nPromotion statistics:")
print(store_53_grocery["onpromotion"].describe())

print("\nRecords around the promotion peak:")
print(
    store_53_grocery[
        (store_53_grocery["date"] >= "2016-04-25") &
        (store_53_grocery["date"] <= "2016-06-05")
    ][["date", "sales", "onpromotion"]].to_string(index=False)
)


Number of records: 1684

Promotion statistics:
count    1684.000000
mean       22.686461
std        33.467577
min         0.000000
25%         0.000000
50%         7.500000
75%        36.000000
max       169.000000
Name: onpromotion, dtype: float64

Records around the promotion peak:
      date  sales  onpromotion
2016-04-25 4436.0           43
2016-04-26 3115.0           35
2016-04-27 3791.0           59
2016-04-28 3009.0           66
2016-04-29 3420.0          169
2016-04-30 5948.0          169
2016-05-01 6712.0          169
2016-05-02 4643.0          169
2016-05-03 4096.0          169
2016-05-04 5109.0          169
2016-05-05 5393.0          169
2016-05-06 4136.0          169
2016-05-07 6825.0          169
2016-05-08 4521.0          169
2016-05-09 5284.0          169
2016-05-10 3886.0          169
2016-05-11 3757.0          169
2016-05-12 3378.0          169
2016-05-13 3468.0          169
2016-05-14 6262.0          169
2016-05-15 6154.0          169
2016-05-16 4030.0          169
20

In [13]:

train = clean_data["train"]

comparison = train[
    (train["family"] == "GROCERY I") &
    (train["date"].between("2016-04-25", "2016-06-05"))
]

print(
    comparison.groupby("date")["onpromotion"]
    .agg(["min", "median", "max"])
    .to_string()
)


            min  median  max
date                        
2016-04-25    0    32.5   55
2016-04-26    0    31.5   59
2016-04-27    0    47.5   86
2016-04-28    0    47.0   83
2016-04-29    0    48.0  169
2016-04-30    0    53.5  169
2016-05-01    0    58.5  169
2016-05-02    0    51.5  169
2016-05-03    0    62.5  169
2016-05-04    0    63.0  169
2016-05-05    0    61.5  169
2016-05-06    0    61.5  169
2016-05-07    0    61.0  169
2016-05-08    0    57.0  169
2016-05-09    0    59.0  169
2016-05-10    0    58.5  169
2016-05-11    0    46.0  169
2016-05-12    0    46.0  169
2016-05-13    0    39.5  169
2016-05-14    0    42.5  169
2016-05-15    0    42.0  169
2016-05-16    0    39.0  169
2016-05-17    0    38.5  169
2016-05-18    0    41.0  169
2016-05-19    0    41.0  169
2016-05-20    0    42.0  169
2016-05-21    0    41.5  169
2016-05-22    0    44.0  169
2016-05-23    0    39.0  169
2016-05-24    0    38.5  169
2016-05-25    0    44.0  169
2016-05-26    0    58.5  169
2016-05-27    

In [14]:
train = clean_data["train"]

promotion_spikes = train[
    (train["family"] == "GROCERY I") &
    (train["date"].between("2016-04-29", "2016-05-31")) &
    (train["onpromotion"] > 100)
]

print(
    promotion_spikes.groupby("store_nbr")
    .agg(
        records=("onpromotion", "count"),
        max_promotion=("onpromotion", "max"),
        avg_promotion=("onpromotion", "mean")
    )
    .sort_values("max_promotion", ascending=False)
    .to_string()
)

           records  max_promotion  avg_promotion
store_nbr                                       
54              33            169     169.000000
53              33            169     169.000000
45               6            146     133.166667
48               6            145     128.166667
47               6            142     133.166667
46               6            138     129.000000
44               6            136     127.833333
49               5            130     123.800000
3                5            125     116.400000
6                4            123     111.250000
50               5            122     117.200000
8                3            117     109.666667
51               2            116     113.500000
5                3            113     106.666667
4                2            108     105.000000
24               1            105     105.000000
38               1            104     104.000000


In [15]:
train = clean_data["train"]

comparison = train[
    (train["family"] == "GROCERY I") &
    (train["store_nbr"].isin([53, 54])) &
    (train["date"].between("2016-04-20", "2016-06-10"))
]

print(
    comparison.pivot(
        index="date",
        columns="store_nbr",
        values="onpromotion"
    ).to_string()
)

store_nbr    53   54
date                
2016-04-20   33   27
2016-04-21   32   19
2016-04-22   39   26
2016-04-23   43   24
2016-04-24   38   36
2016-04-25   43   21
2016-04-26   35   16
2016-04-27   59   28
2016-04-28   66   20
2016-04-29  169  169
2016-04-30  169  169
2016-05-01  169  169
2016-05-02  169  169
2016-05-03  169  169
2016-05-04  169  169
2016-05-05  169  169
2016-05-06  169  169
2016-05-07  169  169
2016-05-08  169  169
2016-05-09  169  169
2016-05-10  169  169
2016-05-11  169  169
2016-05-12  169  169
2016-05-13  169  169
2016-05-14  169  169
2016-05-15  169  169
2016-05-16  169  169
2016-05-17  169  169
2016-05-18  169  169
2016-05-19  169  169
2016-05-20  169  169
2016-05-21  169  169
2016-05-22  169  169
2016-05-23  169  169
2016-05-24  169  169
2016-05-25  169  169
2016-05-26  169  169
2016-05-27  169  169
2016-05-28  169  169
2016-05-29  169  169
2016-05-30  169  169
2016-05-31  169  169
2016-06-01   84   43
2016-06-02   76   26
2016-06-03   68   18
2016-06-04   

In [16]:
train = clean_data["train"]

sales_comparison = train[
    (train["family"] == "GROCERY I") &
    (train["store_nbr"].isin([53, 54])) &
    (train["date"].between("2016-04-20", "2016-06-10"))
]

print(
    sales_comparison.pivot(
        index="date",
        columns="store_nbr",
        values="sales"
    ).to_string()
)

store_nbr       53      54
date                      
2016-04-20  7599.0  5129.0
2016-04-21  6949.0  3807.0
2016-04-22  4102.0  3167.0
2016-04-23  5413.0  3687.0
2016-04-24  4053.0  8456.0
2016-04-25  4436.0  2520.0
2016-04-26  3115.0  2566.0
2016-04-27  3791.0  2051.0
2016-04-28  3009.0  1964.0
2016-04-29  3420.0  2225.0
2016-04-30  5948.0  3744.0
2016-05-01  6712.0  5523.0
2016-05-02  4643.0  3273.0
2016-05-03  4096.0  2907.0
2016-05-04  5109.0  2445.0
2016-05-05  5393.0  2004.0
2016-05-06  4136.0  2282.0
2016-05-07  6825.0  3587.0
2016-05-08  4521.0  3903.0
2016-05-09  5284.0  3473.0
2016-05-10  3886.0  2625.0
2016-05-11  3757.0  2099.0
2016-05-12  3378.0  1762.0
2016-05-13  3468.0  4485.0
2016-05-14  6262.0  3279.0
2016-05-15  6154.0  5608.0
2016-05-16  4030.0  2712.0
2016-05-17  4221.0  3063.0
2016-05-18  3570.0   583.0
2016-05-19  4707.0  2423.0
2016-05-20  4303.0  4321.0
2016-05-21  5800.0  3424.0
2016-05-22  6044.0  4723.0
2016-05-23  3693.0  2669.0
2016-05-24  4132.0  3582.0
2

In [17]:
stores = clean_data["stores"]

print(
    stores[stores["store_nbr"].isin([53, 54])]
    .to_string(index=False)
)

 store_nbr      city  state type  cluster
        53     Manta Manabi    D       13
        54 El Carmen Manabi    C        3


In [18]:
promotion_values = train.loc[
    train["onpromotion"] > 0, "onpromotion"
]

q1 = promotion_values.quantile(0.25)
q3 = promotion_values.quantile(0.75)

iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Upper bound:", upper_bound)

Q1: 1.0
Q3: 12.0
IQR: 11.0
Upper bound: 28.5


In [19]:
train = clean_data["train"]

above_iqr = train[train["onpromotion"] > 28.5]

print("Rows above IQR cap:", len(above_iqr))
print("Percentage:", len(above_iqr) / len(train) * 100)
print("Maximum:", above_iqr["onpromotion"].max())

Rows above IQR cap: 79142
Percentage: 2.6372860300017864
Maximum: 169


In [20]:
promotion_values = train.loc[
    train["onpromotion"] > 0, "onpromotion"
]

for percentile in [0.95, 0.99, 0.995, 0.999]:
    print(
        f"{percentile * 100:.1f}th percentile:",
        promotion_values.quantile(percentile)
    )

95.0th percentile: 55.0
99.0th percentile: 120.0
99.5th percentile: 169.0
99.9th percentile: 169.0


In [21]:
cap = 169

above_cap = train[train["onpromotion"] > cap]

print("Rows above cap:", len(above_cap))
print("Percentage:", len(above_cap) / len(train) * 100)
print("Maximum before:", train["onpromotion"].max())

Rows above cap: 0
Percentage: 0.0
Maximum before: 169


In [22]:
train_clean = clean_data["train"]

print("Maximum promotion:", train_clean["onpromotion"].max())
print(
    "Rows at cap:",
    (train_clean["onpromotion"] == 169).sum()
)

Maximum promotion: 169
Rows at cap: 3059


In [23]:
train = clean_data["train"]

print("Number of product families:", train["family"].nunique())
print("\nProduct families:")
print(train["family"].unique())

Number of product families: 33

Product families:
<StringArray>
[                'AUTOMOTIVE',                  'BABY CARE',
                     'BEAUTY',                  'BEVERAGES',
                      'BOOKS',               'BREAD/BAKERY',
                'CELEBRATION',                   'CLEANING',
                      'DAIRY',                       'DELI',
                       'EGGS',               'FROZEN FOODS',
                  'GROCERY I',                 'GROCERY II',
                   'HARDWARE',         'HOME AND KITCHEN I',
        'HOME AND KITCHEN II',            'HOME APPLIANCES',
                  'HOME CARE',                 'LADIESWEAR',
            'LAWN AND GARDEN',                   'LINGERIE',
           'LIQUOR,WINE,BEER',                  'MAGAZINES',
                      'MEATS',              'PERSONAL CARE',
               'PET SUPPLIES',    'PLAYERS AND ELECTRONICS',
                    'POULTRY',             'PREPARED FOODS',
                    '

In [24]:
train = clean_data["train"]
test = clean_data["test"]

train_families = set(train["family"].unique())
test_families = set(test["family"].unique())

print("Families only in train:")
print(train_families - test_families)

print("\nFamilies only in test:")
print(test_families - train_families)

Families only in train:
set()

Families only in test:
set()


In [25]:
stores = clean_data["stores"]

for column in ["city", "state", "type", "cluster"]:
    print(f"\n{column.upper()}")
    print("Unique values:", stores[column].nunique())
    print(sorted(stores[column].unique()))


CITY
Unique values: 22
['Ambato', 'Babahoyo', 'Cayambe', 'Cuenca', 'Daule', 'El Carmen', 'Esmeraldas', 'Guaranda', 'Guayaquil', 'Ibarra', 'Latacunga', 'Libertad', 'Loja', 'Machala', 'Manta', 'Playas', 'Puyo', 'Quevedo', 'Quito', 'Riobamba', 'Salinas', 'Santo Domingo']

STATE
Unique values: 16
['Azuay', 'Bolivar', 'Chimborazo', 'Cotopaxi', 'El Oro', 'Esmeraldas', 'Guayas', 'Imbabura', 'Loja', 'Los Rios', 'Manabi', 'Pastaza', 'Pichincha', 'Santa Elena', 'Santo Domingo de los Tsachilas', 'Tungurahua']

TYPE
Unique values: 5
['A', 'B', 'C', 'D', 'E']

CLUSTER
Unique values: 17
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17)]


In [26]:
holidays = clean_data["holidays_events"]

for column in ["type", "locale", "locale_name", "transferred"]:
    print(f"\n{column.upper()}")
    print("Unique values:", holidays[column].nunique())
    print(holidays[column].value_counts(dropna=False).to_string())


TYPE
Unique values: 6
type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5

LOCALE
Unique values: 3
locale
National    174
Local       152
Regional     24

LOCALE_NAME
Unique values: 24
locale_name
Ecuador                           174
Quito                              13
Riobamba                           12
Guaranda                           12
Latacunga                          12
Ambato                             12
Guayaquil                          11
Cuenca                              7
Ibarra                              7
Manta                               6
Cotopaxi                            6
Libertad                            6
Puyo                                6
Imbabura                            6
Machala                             6
Santo Domingo                       6
El Carmen                           6
Cayambe                             6
Esmeraldas                          6
Quevedo            

In [27]:
transactions = clean_data["transactions"]

print("Transactions dtypes:")
print(transactions.dtypes)

print("\nTransactions summary:")
print(transactions["transactions"].describe())

print("\nNegative transaction counts:")
print((transactions["transactions"] < 0).sum())

print("\nZero transaction counts:")
print((transactions["transactions"] == 0).sum())


Transactions dtypes:
date            datetime64[us]
store_nbr                int64
transactions             int64
dtype: object

Transactions summary:
count    83488.000000
mean      1694.602158
std        963.286644
min          5.000000
25%       1046.000000
50%       1393.000000
75%       2079.000000
max       8359.000000
Name: transactions, dtype: float64

Negative transaction counts:
0

Zero transaction counts:
0


In [28]:
oil = clean_data["oil"]

print("Oil price summary:")
print(oil["dcoilwtico"].describe())

print("\nZero oil prices:", (oil["dcoilwtico"] == 0).sum())
print("Negative oil prices:", (oil["dcoilwtico"] < 0).sum())
print("Missing oil prices:", oil["dcoilwtico"].isna().sum())

Oil price summary:
count    1218.000000
mean       67.692159
std        25.629744
min        26.190000
25%        46.422500
50%        53.200000
75%        95.685000
max       110.620000
Name: dcoilwtico, dtype: float64

Zero oil prices: 0
Negative oil prices: 0
Missing oil prices: 0


In [29]:
for name, df in clean_data.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())


STORES
Shape: (54, 5)
Missing values: 0
Duplicate rows: 0

TRAIN
Shape: (3000888, 6)
Missing values: 0
Duplicate rows: 0

OIL
Shape: (1218, 2)
Missing values: 0
Duplicate rows: 0

HOLIDAYS_EVENTS
Shape: (350, 6)
Missing values: 0
Duplicate rows: 0

TRANSACTIONS
Shape: (83488, 3)
Missing values: 0
Duplicate rows: 0

TEST
Shape: (28512, 5)
Missing values: 0
Duplicate rows: 0
